<a href="https://colab.research.google.com/github/phucsz/DAAI_N1.4/blob/main/Shippers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Đọc và kiểm tra cấu trúc dữ liệu Shipper từ file shipments_realistic.csv

In [10]:
import pandas as pd

# Đọc dữ liệu shipments_realistic.csv để trích xuất các cột về shipper
shipments_df = pd.read_csv('/content/shipments_realistic.csv')

# Lọc ra các cột liên quan đến shipper như yêu cầu
shipper_cols = [
    'shipper', 'shipper_name', 'shipper_phone', 'shipper_gender',
    'shipper_age', 'shipper_martial_status', 'shipper_education',
    'shipper_company', 'shipper_vehicle', 'shipper_experience_years',
    'join_date', 'working_shift', 'city', 'region', 'district',
    'shipper_rating', 'delivery_success_rate', 'average_delivery_time'
]

# Lấy các cột tồn tại trong file
available_cols = [col for col in shipper_cols if col in shipments_df.columns]
print("Các cột shipper tìm thấy:", available_cols)

# Tạo dataframe chứa thông tin shipper duy nhất
shippers_raw = shipments_df[available_cols].drop_duplicates().reset_index(drop=True)

print("\n--- THÔNG TIN CẤU TRÚC BAN ĐẦU CỦA SHIPPERS ---")
shippers_raw.info()

print("\n--- SỐ LƯỢNG GIÁ TRỊ NULL TRONG MỖI CỘT ---")
print(shippers_raw.isnull().sum())

display(shippers_raw.head())

Các cột shipper tìm thấy: ['shipper_name', 'shipper_phone', 'shipper_gender', 'shipper_age', 'shipper_education', 'shipper_company', 'shipper_vehicle', 'shipper_experience_years', 'join_date', 'working_shift', 'city', 'region', 'district', 'shipper_rating', 'delivery_success_rate', 'average_delivery_time']

--- THÔNG TIN CẤU TRÚC BAN ĐẦU CỦA SHIPPERS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   shipper_name              80 non-null     object 
 1   shipper_phone             80 non-null     int64  
 2   shipper_gender            80 non-null     object 
 3   shipper_age               80 non-null     int64  
 4   shipper_education         80 non-null     object 
 5   shipper_company           80 non-null     object 
 6   shipper_vehicle           80 non-null     object 
 7   shipper_experience_years  80 non-null   

,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_education,shipper_company,shipper_vehicle,shipper_experience_years,join_date,working_shift,city,region,district,shipper_rating,delivery_success_rate,average_delivery_time
0,Bùi Văn Long,991476209,Male,27,Bachelor,Viettel Post,Truck,7,2026-03-17,Evening,Phan Rang-Thap Cham,Central,District #25,5.0,99.0,61
1,Trần Anh Khánh,959297982,Male,41,Bachelor,J&T Express,Van,2,2025-01-29,Afternoon,Phan Thiet,Central,District #29,4.9,98.4,72
2,Hoàng Thị Khánh,927142576,Male,30,High School,GHN,Motorbike,10,2019-11-13,Evening,Long Xuyen,West,District #34,4.8,95.1,53
3,Trần Đức Vy,971617475,Female,42,College,Viettel Post,Truck,8,2025-12-22,Evening,Kon Tum,Central,District #27,5.0,96.3,53
4,Trần Minh Cường,979196342,Male,31,College,BEST Express,Truck,10,2019-12-19,Morning,Da Nang,Central,District #23,4.6,95.7,62


### 2. Xử lý dữ liệu trùng lặp, khuyết thiếu (Null/NaN) và tiến hành chuẩn hóa 3NF
Để đạt chuẩn hóa 3NF, loại bỏ dư thừa dữ liệu và phụ thuộc bắc cầu:
- Các thông tin địa lý (`city`, `region`, `district`) nên được tách thành bảng danh mục địa lý độc lập.
- Các thông tin lặp đi lặp lại dạng chuỗi như thông tin công ty vận chuyển (`shipper_company`), phương tiện (`shipper_vehicle`), ca làm việc (`working_shift`) sẽ được tách thành các bảng danh mục (Dim) riêng và ánh xạ khóa ngoại vào bảng chính.

In [12]:
# 1. Xử lý giá trị khuyết thiếu (nếu có)
# Đối với các cột số lượng, điền median hoặc mean, đối với phân loại điền 'Unknown'
for col in shippers_raw.columns:
    if shippers_raw[col].isnull().any():
        if shippers_raw[col].dtype in ['int64', 'float64']:
            shippers_raw[col] = shippers_raw[col].fillna(shippers_raw[col].median())
        else:
            shippers_raw[col] = shippers_raw[col].fillna('Unknown')

# Lấy trường định danh duy nhất cho shipper.
# Trong shipments_df gốc có 'shipper_id', hãy kết nối lại hoặc tạo ID duy nhất dựa trên shipper_phone/shipper_name
# Ở đây ta sẽ lấy shipper_id từ shipments_df ban đầu ghép vào shippers_raw
shippers_with_id = shipments_df[['shipper_id'] + available_cols].drop_duplicates(subset=['shipper_id']).reset_index(drop=True)

# Đảm bảo loại bỏ hoàn toàn các bản ghi trùng lặp trên khóa chính 'shipper_id'
shippers_cleaned = shippers_with_id.copy()

# 2. Tạo bảng danh mục Địa lý (Geography Dim) để chuẩn hóa 3NF
geo_cols = ['region', 'city', 'district']
if all(col in shippers_cleaned.columns for col in geo_cols):
    geo_dim = shippers_cleaned[geo_cols].drop_duplicates().reset_index(drop=True)
    geo_dim['geo_id'] = [f'GEO-{i+1:03d}' for i in range(len(geo_dim))]

    # Ánh xạ geo_id vào bảng shippers chính và loại bỏ các cột địa lý cũ
    shippers_cleaned = shippers_cleaned.merge(geo_dim, on=geo_cols, how='left')
    shippers_cleaned = shippers_cleaned.drop(columns=geo_cols)

    # Xuất file Geography
    geo_dim.to_csv('/content/shipper_geography_dim.csv', index=False)
    print("Đã lưu: /content/shipper_geography_dim.csv")
    display(geo_dim.head())

# 3. Tạo bảng danh mục Công ty vận chuyển (Company Dim)
if 'shipper_company' in shippers_cleaned.columns:
    companies = shippers_cleaned['shipper_company'].unique()
    company_dim = pd.DataFrame({
        'company_id': [f'CPN-{i+1:02d}' for i in range(len(companies))],
        'shipper_company': companies
    })
    shippers_cleaned = shippers_cleaned.merge(company_dim, on='shipper_company', how='left')
    shippers_cleaned = shippers_cleaned.drop(columns=['shipper_company'])

    company_dim.to_csv('/content/shipper_companies_dim.csv', index=False)
    print("Đã lưu: /content/shipper_companies_dim.csv")

# 4. Xuất bảng Shippers chính đã chuẩn hóa 3NF
shippers_cleaned.to_csv('/content/shippers_3nf.csv', index=False)
print("\nĐã lưu bảng chính: /content/shippers_3nf.csv")

print("\n--- 5 DÒNG ĐẦU BẢNG SHIPPERS CHUẨN 3NF ---")
display(shippers_cleaned.head())

Đã lưu: /content/shipper_geography_dim.csv


,region,city,district,geo_id
0,Central,Phan Rang-Thap Cham,District #25,GEO-001
1,Central,Phan Thiet,District #29,GEO-002
2,West,Long Xuyen,District #34,GEO-003
3,Central,Kon Tum,District #27,GEO-004
4,Central,Da Nang,District #23,GEO-005


Đã lưu: /content/shipper_companies_dim.csv

Đã lưu bảng chính: /content/shippers_3nf.csv

--- 5 DÒNG ĐẦU BẢNG SHIPPERS CHUẨN 3NF ---


,shipper_id,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_education,shipper_vehicle,shipper_experience_years,join_date,working_shift,shipper_rating,delivery_success_rate,average_delivery_time,geo_id,company_id
0,SHP00001,Bùi Văn Long,991476209,Male,27,Bachelor,Truck,7,2026-03-17,Evening,5.0,99.0,61,GEO-001,CPN-01
1,SHP00002,Trần Anh Khánh,959297982,Male,41,Bachelor,Van,2,2025-01-29,Afternoon,4.9,98.4,72,GEO-002,CPN-02
2,SHP00003,Hoàng Thị Khánh,927142576,Male,30,High School,Motorbike,10,2019-11-13,Evening,4.8,95.1,53,GEO-003,CPN-03
3,SHP00004,Trần Đức Vy,971617475,Female,42,College,Truck,8,2025-12-22,Evening,5.0,96.3,53,GEO-004,CPN-01
4,SHP00005,Trần Minh Cường,979196342,Male,31,College,Truck,10,2019-12-19,Morning,4.6,95.7,62,GEO-005,CPN-04
